In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.phase2_cleaning import detect_duplicates

In [2]:
duplicates = detect_duplicates(
    PROJECT_ROOT,
    workers=4,
)

duplicates.head(30)

Image hashing: 1,000/42,541
Image hashing: 2,000/42,541
Image hashing: 3,000/42,541
Image hashing: 4,000/42,541
Image hashing: 5,000/42,541
Image hashing: 6,000/42,541
Image hashing: 7,000/42,541
Image hashing: 8,000/42,541
Image hashing: 9,000/42,541
Image hashing: 10,000/42,541
Image hashing: 11,000/42,541
Image hashing: 12,000/42,541
Image hashing: 13,000/42,541
Image hashing: 14,000/42,541
Image hashing: 15,000/42,541
Image hashing: 16,000/42,541
Image hashing: 17,000/42,541
Image hashing: 18,000/42,541
Image hashing: 19,000/42,541
Image hashing: 20,000/42,541
Image hashing: 21,000/42,541
Image hashing: 22,000/42,541
Image hashing: 23,000/42,541
Image hashing: 24,000/42,541
Image hashing: 25,000/42,541
Image hashing: 26,000/42,541
Image hashing: 27,000/42,541
Image hashing: 28,000/42,541
Image hashing: 29,000/42,541
Image hashing: 30,000/42,541
Image hashing: 31,000/42,541
Image hashing: 32,000/42,541
Image hashing: 33,000/42,541
Image hashing: 34,000/42,541
Image hashing: 35,000/4

,duplicate_group,image_key,image_id,dataset,duplicate_type,review_status
0,duplicate_00001,aptos__c8827d05711175da,001639a390f0,aptos,near_perceptual_hash,pending_review
1,duplicate_00001,aptos__11b2efaae2c39263,005b95c28852,aptos,near_perceptual_hash,pending_review
2,duplicate_00001,aptos__f9409e06017fe46c,002c21358ce6,aptos,near_perceptual_hash,pending_review
3,duplicate_00001,aptos__2b8228ecc008b575,0024cdab0c1e,aptos,near_perceptual_hash,pending_review
4,duplicate_00001,aptos__8a58801bb4e89fda,000c1434d8d7,aptos,near_perceptual_hash,pending_review
5,duplicate_00001,aptos__56f97594ad38fa59,00a8624548a9,aptos,near_perceptual_hash,pending_review
6,duplicate_00001,aptos__9b0b1f3110f4a445,0097f532ac9f,aptos,near_perceptual_hash,pending_review
7,duplicate_00001,aptos__2bc5841cf5eee5f8,00cb6555d108,aptos,exact_sha256;near_perceptual_hash,pending_review
8,duplicate_00001,aptos__8229f86f94c8c7e2,00cc2b75cddd,aptos,near_perceptual_hash,pending_review
9,duplicate_00001,aptos__6ce14f7a48048daf,00f6c1be5a33,aptos,near_perceptual_hash,pending_review


In [3]:
import importlib
import src.phase2_cleaning

importlib.reload(src.phase2_cleaning)

from src.phase2_cleaning import detect_duplicates

duplicates = detect_duplicates(
    PROJECT_ROOT,
    workers=4,
)

duplicates.head(30)

Reusing existing image_hashes.csv; no images will be hashed again.
Exact-duplicate images: 544
Exact-duplicate groups: 266
Perceptual-hash matches were saved for manual review only.


,duplicate_group,image_key,image_id,dataset,duplicate_type,review_status
0,exact_016d1670c190,aptos__b0f3806adb3f7235,530d78467615,aptos,exact_sha256,automatic_exact_match
1,exact_016d1670c190,aptos__00cf743257d485ed,c1c8550508e0,aptos,exact_sha256,automatic_exact_match
2,exact_01f04eb1f0de,aptos__41d4aaeb0a1dc62b,8d3d67661620,aptos,exact_sha256,automatic_exact_match
3,exact_01f04eb1f0de,aptos__35a3aebba1b6b791,1248275c3f91,aptos,exact_sha256,automatic_exact_match
4,exact_022b0e0373dd,aptos__6c7510d3d6781285,0ac436400db4,aptos,exact_sha256,automatic_exact_match
5,exact_022b0e0373dd,aptos__6a7c5011a3db4b4f,fda39982a810,aptos,exact_sha256,automatic_exact_match
6,exact_032dc113c5f9,aptos__7b053c0edaed6e6e,a49b0b4484ea,aptos,exact_sha256,automatic_exact_match
7,exact_032dc113c5f9,aptos__59600e68271eba7a,be71e340dbaa,aptos,exact_sha256,automatic_exact_match
8,exact_0351e83371a7,aptos__633c9cba342deefe,ad570b850a4f,aptos,exact_sha256,automatic_exact_match
9,exact_0351e83371a7,aptos__57cb2c73b88c157b,4f985a7320d9,aptos,exact_sha256,automatic_exact_match


In [5]:
import pandas as pd
hashes = pd.read_csv(
    PROJECT_ROOT / "reports/tables/image_hashes.csv"
)

metadata = pd.read_csv(
    PROJECT_ROOT / "data/metadata/unified_metadata.csv"
)

summary = pd.read_csv(
    PROJECT_ROOT
    / "reports/tables/perceptual_hash_review_summary.csv"
)

# Review only the 20 largest perceptual-hash collision groups.
top_hashes = summary.head(20)["perceptual_hash"]

review = (
    hashes.loc[
        hashes["perceptual_hash"].isin(top_hashes)
    ]
    .merge(
        metadata[
            [
                "image_key",
                "image_id",
                "dataset",
                "image_path",
                "processed_path",
                "model_split",
            ]
        ],
        on="image_key",
        how="left",
    )
    .sort_values("perceptual_hash")
)

review["manual_decision"] = "not_reviewed"
review["action"] = "do_not_remove_automatically"

review.to_csv(
    PROJECT_ROOT
    / "reports/tables/perceptual_hash_manual_review.csv",
    index=False,
)

review.head(30)

C:\Users\poorv\AppData\Local\Temp\ipykernel_2788\3908796947.py:6: DtypeWarning: Columns (0: official_split, 1: quality_label_source, 2: duplicate_group, 3: duplicate_type, 4: review_status, 5: dme_label_source, 6: dme_label_scheme, 7: lesion_annotation_type) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(


,image_key,image_id_x,dataset_x,sha256,perceptual_hash,image_id_y,dataset_y,image_path,processed_path,model_split,manual_decision,action
1020,eyepacs__ab6caf9ec3556759,15873_left,eyepacs,6775d5b57affc7bc76c2dd311a1d9182ae696dc51c60df...,3070e0e0e0e07070,15873_left,eyepacs,data/raw/eyepacs/data/15873_left.jpeg,data/processed/images/eyepacs/eyepacs__ab6caf9...,train,not_reviewed,do_not_remove_automatically
1726,eyepacs__b42dad2b1988207c,19928_left,eyepacs,795a9c0f56935a06b145f7e36a7f42387b6e4a370edd9d...,3070e0e0e0e07070,19928_left,eyepacs,data/raw/eyepacs/data/19928_left.jpeg,data/processed/images/eyepacs/eyepacs__b42dad2...,train,not_reviewed,do_not_remove_automatically
2231,eyepacs__8f01946c6ed0fd8e,22920_left,eyepacs,aa93f2679328132c984020a96e3ba87665b0d2cd7e6d63...,3070e0e0e0e07070,22920_left,eyepacs,data/raw/eyepacs/data/22920_left.jpeg,data/processed/images/eyepacs/eyepacs__8f01946...,train,not_reviewed,do_not_remove_automatically
3923,eyepacs__6ad4c1cee483765a,32758_left,eyepacs,e435de77b4160d92d7691b1c3c700175c7389dcda31c8e...,3070e0e0e0e07070,32758_left,eyepacs,data/raw/eyepacs/data/32758_left.jpeg,data/processed/images/eyepacs/eyepacs__6ad4c1c...,train,not_reviewed,do_not_remove_automatically
6029,eyepacs__3cd9a48fbb87881e,4805_left,eyepacs,b9d836453c28eae1085dbd1d98fa8574cd268472fe51eb...,3070e0e0e0e07070,4805_left,eyepacs,data/raw/eyepacs/data/4805_left.jpeg,data/processed/images/eyepacs/eyepacs__3cd9a48...,train,not_reviewed,do_not_remove_automatically
4376,eyepacs__701c1235efc31733,35464_left,eyepacs,dae9517820366bed3dd8ac3c7dfd624ffedc6a7fd17239...,3070e0e0e0e07070,35464_left,eyepacs,data/raw/eyepacs/data/35464_left.jpeg,data/processed/images/eyepacs/eyepacs__701c123...,train,not_reviewed,do_not_remove_automatically
3928,eyepacs__e4dac3f43d2d670b,32787_left,eyepacs,2564ab005fcba45286b91e11c226558237de05d4c41f46...,3070e0e0e0e07070,32787_left,eyepacs,data/raw/eyepacs/data/32787_left.jpeg,data/processed/images/eyepacs/eyepacs__e4dac3f...,train,not_reviewed,do_not_remove_automatically
5393,eyepacs__4cb55ae252af2c21,41080_left,eyepacs,2550f86b125ef9022a644c0ab0415b57f35096a671c16a...,3070e0e0e0e07070,41080_left,eyepacs,data/raw/eyepacs/data/41080_left.jpeg,data/processed/images/eyepacs/eyepacs__4cb55ae...,train,not_reviewed,do_not_remove_automatically
6024,eyepacs__745726aeb13fc457,4784_left,eyepacs,549c6d962e9df0223171a12d7b2a09a13eada90a31951a...,3070e0e0e0e07070,4784_left,eyepacs,data/raw/eyepacs/data/4784_left.jpeg,data/processed/images/eyepacs/eyepacs__745726a...,validation,not_reviewed,do_not_remove_automatically
1478,eyepacs__a132272950771fae,18473_left,eyepacs,5fbca837d968b062e08a0cab5f8a48c1cac07fc228b06e...,3070e0e0e0e07070,18473_left,eyepacs,data/raw/eyepacs/data/18473_left.jpeg,data/processed/images/eyepacs/eyepacs__a132272...,train,not_reviewed,do_not_remove_automatically
